# 第五课：交通图与空间消息传递

前四课只处理时间模式。本课回答最核心的空间问题：**一个传感器怎样使用邻近传感器的信息？**

完成后你应该能解释：

1. 速度矩阵和邻接矩阵如何通过节点顺序对齐；
2. `A[i,j]` 的边权表示什么；
3. 为什么图卷积的第一步是 `A @ X`；
4. 自环和邻接归一化各自解决什么问题；
5. 图卷积为何能处理不同节点数的城市。

## 0. 先建立两个矩阵的对应关系

- 速度矩阵 `V[t,n]`：时间 `t`、第 `n` 个传感器的速度；
- 邻接矩阵 `A[i,j]`：传感器 `i` 与 `j` 的空间连接权重。

两者靠**相同的节点顺序**连接：`V[:, n]` 和 `A[n, :]` 必须指向同一个传感器。图文件中的 sensor IDs 不是模型参数，但它们是数据对齐的重要元数据。

In [ ]:
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.data.graph import load_adjacency, normalize_adjacency  # noqa: E402
from crosscity.models.stgcn import GraphConv  # noqa: E402

config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
raw_a = load_adjacency(config.dataset.adjacency_path)
print('速度矩阵:', bundle.raw_values.shape)
print('邻接矩阵:', raw_a.shape)
assert raw_a.shape == (bundle.raw_values.shape[1],) * 2

上面的断言只验证节点数量一致；更严格的数据准备还应核对传感器 ID 和列名。预处理版 `adj_mx.pkl` 通常保存 `(sensor_ids, id_to_index, adjacency)`。

In [ ]:
with Path(config.dataset.adjacency_path).open('rb') as handle:
    payload = pickle.load(handle, encoding='latin1')

if isinstance(payload, (tuple, list)) and len(payload) >= 3:
    sensor_ids, id_to_index, adjacency_from_payload = payload[:3]
    print('前 5 个 sensor ID:', list(sensor_ids[:5]))
    print('ID → index 示例:', list(id_to_index.items())[:3])
    print('ID 数量:', len(sensor_ids))
    assert all(id_to_index[sensor_id] == i for i, sensor_id in enumerate(sensor_ids))
    assert np.allclose(raw_a, adjacency_from_payload)
else:
    sensor_ids = np.arange(raw_a.shape[0])
    print('该图文件只包含邻接矩阵，没有 sensor ID 元数据。')

## 1. 先认识真实交通图

边权不是速度相关系数。本数据的图由道路网络中的传感器距离经过核函数转换并截断得到：距离近、道路上相连的节点权重大；没有保留的连接权重为 0。

In [ ]:
positive = raw_a[raw_a > 0]
degree = (raw_a > 0).sum(axis=1)
weighted_degree = raw_a.sum(axis=1)
summary = {
    'nodes': raw_a.shape[0],
    'nonzero entries': int((raw_a > 0).sum()),
    'density': float((raw_a > 0).mean()),
    'symmetric': bool(np.allclose(raw_a, raw_a.T)),
    'self-loop count in raw A': int(np.count_nonzero(np.diag(raw_a))),
    'positive weight min/mean/max': (positive.min(), positive.mean(), positive.max()),
}
pd.Series(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
image = axes[0].imshow(raw_a, cmap='viridis', aspect='auto')
axes[0].set(title='Raw adjacency matrix', xlabel='source j', ylabel='target i')
fig.colorbar(image, ax=axes[0], fraction=0.046)
axes[1].hist(degree, bins=range(int(degree.max()) + 2), align='left')
axes[1].set(title='Number of neighbors', xlabel='degree', ylabel='nodes')
axes[2].hist(weighted_degree, bins=20)
axes[2].set(title='Weighted degree', xlabel='sum of edge weights')
plt.tight_layout()
plt.show()

邻接矩阵图像看起来稀疏是正常的：一个道路传感器只应接收少数局部邻居，而不是全城所有节点。如果原始图不对称，则 `A[i,j]` 与 `A[j,i]` 可不同，表示道路传播方向不同；当前实现会保留这种方向性。

## 2. 查看一个真实节点的邻居

我们选连接数较多的节点，打印它从哪些节点接收消息。注意：没有经纬度文件时只能画拓扑示意图，不能把它误称为真实地图。

In [ ]:
node = int(np.argmax(degree))
neighbors = np.flatnonzero(raw_a[node] > 0)
neighbor_table = pd.DataFrame({
    'index': neighbors,
    'sensor_id': [sensor_ids[i] for i in neighbors],
    'A[node, neighbor]': raw_a[node, neighbors],
}).sort_values('A[node, neighbor]', ascending=False)
print('中心节点 index / ID:', node, sensor_ids[node])
neighbor_table

In [ ]:
angles = np.linspace(0, 2 * np.pi, len(neighbors), endpoint=False)
plt.figure(figsize=(6, 6))
for angle, neighbor in zip(angles, neighbors):
    x_pos, y_pos = np.cos(angle), np.sin(angle)
    plt.plot([0, x_pos], [0, y_pos], linewidth=1 + 4 * raw_a[node, neighbor], alpha=.6)
    plt.text(x_pos, y_pos, str(neighbor), ha='center', va='center', bbox={'boxstyle': 'circle', 'fc': 'white'})
plt.scatter([0], [0], s=700, color='tomato', zorder=3)
plt.text(0, 0, str(node), ha='center', va='center')
plt.title('Topological ego graph (not geographic coordinates)')
plt.axis('equal')
plt.axis('off')
plt.show()

## 3. 手算一次邻居聚合

先用一个 4 节点玩具图。若采用 `A @ x`，则

$$z_i=\sum_j A_{ij}x_j$$

第 `i` 行说明节点 `i` 从哪些 `j` 接收多少信息。

In [ ]:
toy_a = torch.tensor([
    [0.0, 1.0, 0.0, 0.0],
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [0.0, 0.0, 1.0, 0.0],
])
toy_speed = torch.tensor([60.0, 40.0, 20.0, 10.0])
print('未经归一化的 A @ x:', toy_a @ toy_speed)
print('节点 1 手算 = 1×60 + 1×20 =', 1 * 60 + 1 * 20)

未经归一化时，邻居越多，结果数值通常越大；而且节点自己的信息消失了。项目采用

$$\hat A=D^{-1/2}(A+I)D^{-1/2}$$

其中 `+I` 添加自环，`D` 是加自环后的度矩阵。对称归一化控制不同度数节点之间的尺度。注意：它通常**不保证每行和为 1**，所以不应简单称为邻居平均值。

In [ ]:
toy_norm = normalize_adjacency(toy_a)
display(pd.DataFrame(toy_norm.numpy()).round(3))
print('各行之和:', toy_norm.sum(dim=1))
print('归一化聚合:', toy_norm @ toy_speed)
print('节点 1 的逐项贡献:', toy_norm[1] * toy_speed)
assert torch.all(torch.diag(toy_norm) > 0), '加入自环后对角线应大于 0'

## 4. 在 METR-LA 的一个真实时刻聚合速度

缺失输入仍按前面课程的规则填充为训练均值对应的标准化 0。这里使用标准化输入，是为了与真正送入 STGCN 的数据完全一致。

In [ ]:
norm_a = normalize_adjacency(raw_a)
x_window, _, _ = bundle.train[1000]
current_z = x_window[-1, :, 0]
aggregated_z = norm_a @ current_z
current_speed = bundle.scaler.inverse_transform(current_z)
aggregated_speed = bundle.scaler.inverse_transform(aggregated_z)

example = pd.DataFrame({
    'sensor': np.arange(len(current_speed)),
    'own speed': current_speed.numpy(),
    'graph-aggregated feature': aggregated_speed.numpy(),
    'difference': (aggregated_speed - current_speed).numpy(),
})
example.iloc[[node, *neighbors[:min(5, len(neighbors))]]]

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(current_speed, aggregated_speed, s=14, alpha=.6)
limits = [min(current_speed.min(), aggregated_speed.min()), max(current_speed.max(), aggregated_speed.max())]
plt.plot(limits, limits, '--', color='gray')
plt.xlabel('Own speed (mph)')
plt.ylabel('Graph-aggregated feature (mph)')
plt.title('A node versus its spatially aggregated context')
plt.show()

聚合值不是新的预测目标，也不一定是严格的物理速度平均值。它是一个空间上下文特征：如果上游邻居已经降速而本节点尚未降速，这种差异可能帮助模型提前识别拥堵传播。后续可学习层决定怎样解释它。

## 5. 从 `A @ X` 到一层图卷积

项目中的简化图卷积为：

$$H'=\mathrm{ReLU}(\hat A H W+b)$$

`A` 负责节点间消息传递，`W,b` 负责通道变换。可学习参数只与通道数有关，不与节点数有关。

In [ ]:
torch.manual_seed(42)
graph_conv = GraphConv(channels=4)
features = torch.randn(2, 4, 12, 207, requires_grad=True)  # [B,C,T,N]
output = graph_conv(features, norm_a)
loss = output.square().mean()
loss.backward()
print('input / output:', features.shape, output.shape)
print('linear weight:', graph_conv.linear.weight.shape)
print('节点数是否出现在参数中:', 207 in graph_conv.linear.weight.shape)
print('输入梯度绝对值均值:', features.grad.abs().mean().item())
print('权重梯度绝对值均值:', graph_conv.linear.weight.grad.abs().mean().item())


由于参数是 `[C_out,C_in]` 而非 `[N,N]`，同一个 `GraphConv` 可以加载到 325 节点城市；推理时换成目标城市自己的 325×325 邻接矩阵。图结构本身不从源城市 checkpoint 搬过去。

In [ ]:
fake_325 = torch.randn(2, 4, 12, 325)
fake_a_325 = normalize_adjacency(torch.eye(325))
with torch.no_grad():
    fake_out = graph_conv(fake_325, fake_a_325)
print('325 节点输出:', fake_out.shape)
assert fake_out.shape == fake_325.shape

## 6. 对照实验：图真的改变了输出吗？

把真实图换成单位阵，相当于每个节点只处理自己。两次使用同一个 `GraphConv`，所以输出差异只能来自消息传递结构。

In [ ]:
graph_conv.eval()
small_features = features.detach()[:1]
identity = torch.eye(207)
with torch.no_grad():
    spatial_output = graph_conv(small_features, norm_a)
    self_only_output = graph_conv(small_features, identity)
difference = (spatial_output - self_only_output).abs()
print('平均绝对差:', difference.mean().item())
print('最大绝对差:', difference.max().item())
assert difference.max() > 0

这个实验只证明图参与了计算，不能证明图改善预测。要证明有效，必须在相同数据、训练预算和随机种子下比较真实图、单位阵、随机图或打乱图的测试误差。

## 7. 进阶性质：节点重编号不应改变语义

若同时重排特征的节点轴和邻接矩阵的行列，图卷积输出应按相同方式重排。这叫 permutation equivariance，也是图网络不依赖传感器编号名称的原因。

In [ ]:
permutation = torch.randperm(207)
permuted_x = small_features[..., permutation]
permuted_a = norm_a[permutation][:, permutation]
with torch.no_grad():
    original_y = graph_conv(small_features, norm_a)
    permuted_y = graph_conv(permuted_x, permuted_a)
error = (permuted_y - original_y[..., permutation]).abs().max().item()
print('同时重排后的最大对应误差:', error)
assert error < 1e-5

## 8. 本课验收问题

请先用自己的话回答：

1. 速度矩阵的第 `n` 列为什么必须与邻接矩阵第 `n` 行/列对应？如果错位会怎样？
2. `A[i,j] > 0` 在 `A @ X` 的约定下表示谁向谁发送信息？
3. 为什么要添加自环？
4. 对称归一化后的每行和是否一定为 1？它与普通加权平均有何不同？
5. 图聚合后的数值为什么不应直接解释为预测速度？
6. `GraphConv` 为什么能从 207 节点迁移到 325 节点？迁移了什么、没有迁移什么？
7. 真实图与单位阵输出不同，能否证明真实图提高了预测性能？为什么？
8. 为什么节点特征和邻接矩阵必须一起重排？

请把以下结果带回来：图是否对称、图密度、节点度数范围、所选中心节点及其邻居数、真实图/单位阵输出平均差，以及你最不确定的一题。

下一课会拆开完整 STGCN：时间门控卷积 → 图卷积 → 时间门控卷积 → 多步预测头，并做真实图、单位阵的受控消融。